# KNN для прогнозирования стоимости поездки

Ноутбук обучает `KNeighborsRegressor` на очищенных данных такси. Он сам загружает датасет, создает признаки, делает train/test split и считает метрики качества.

In [8]:
import os
import time
from pathlib import Path

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET_COL = "total_amount"

PROJECT_ROOT_CANDIDATES = [Path.cwd(), *Path.cwd().parents]
DATA_NAMES = [
    "datasets/short_my_clean_3_with_weather.parquet",
    "datasets_local/short_my_clean_3_with_weather.parquet",
]
DATA_CANDIDATES = [base / name for base in PROJECT_ROOT_CANDIDATES for name in DATA_NAMES]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)
PROJECT_ROOT = next((base for base in PROJECT_ROOT_CANDIDATES if any((base / name).exists() for name in DATA_NAMES)), Path.cwd())

if DATA_PATH is None:
    # Функция для безопасного формирования относительного пути (работает на всех версиях Python)
    def get_display_path(p, root):
        try:
            return str(p.relative_to(root))
        except ValueError:
            return str(p)
            
    checked_paths = "\n".join(f"- {get_display_path(path, PROJECT_ROOT)}" for path in DATA_CANDIDATES)
    raise FileNotFoundError(f"Датасет не найден. Проверенные пути:\n{checked_paths}")

print("Путь к датасету:", DATA_PATH.relative_to(PROJECT_ROOT))

Путь к датасету: datasets\short_my_clean_3_with_weather.parquet


## 1. Загрузка данных

Используется тот же целевой столбец, что и в остальных моделях стоимости: `total_amount`. Даты приводятся к формату datetime, затем оставляются поездки за 2023-2024 годы.

In [9]:
print("Загружаем датасет...")
df = pd.read_parquet(DATA_PATH)

df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")
df = df.loc[
    df["tpep_pickup_datetime"].ge(pd.Timestamp("2023-01-01"))
    & df["tpep_pickup_datetime"].lt(pd.Timestamp("2025-01-01"))
].copy()

print("Размер таблицы:", df.shape)
print("Диапазон дат:", df["tpep_pickup_datetime"].min(), "-", df["tpep_pickup_datetime"].max())
print("Количество поездок по годам:")
print(df["tpep_pickup_datetime"].dt.year.value_counts().sort_index())
df.head()

Загружаем датасет...
Размер таблицы: (100000, 34)
Диапазон дат: 2023-01-01 00:01:58 - 2023-01-31 23:59:47
Количество поездок по годам:
tpep_pickup_datetime
2023    100000
Name: count, dtype: int64


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,DO_Borough,DO_Zone,DO_lon,DO_lat,distance_group,duration_min,temperature,precipitation,snowfall,weather_code
0,2,2023-01-29 17:52:02,2023-01-29 17:56:43,1,1.17,1.0,N,262,74,2,...,Manhattan,East Harlem North,-73.937346,40.801169,short,4.683333,9.6,0.0,0.0,3
1,1,2023-01-08 15:57:24,2023-01-08 16:02:47,1,0.90,1.0,N,229,237,2,...,Manhattan,Upper East Side South,-73.965635,40.768615,very_short,5.383333,4.2,0.0,0.0,3
2,2,2023-01-21 19:38:01,2023-01-21 19:45:02,1,0.95,1.0,N,45,261,1,...,Manhattan,World Trade Center,-74.013023,40.709139,very_short,7.016667,3.1,0.0,0.0,3
3,2,2023-01-23 16:07:31,2023-01-23 16:26:46,5,0.88,1.0,N,237,141,1,...,Manhattan,Lenox Hill West,-73.959635,40.766948,very_short,19.250000,2.9,0.6,0.0,53
4,2,2023-01-26 21:21:08,2023-01-26 21:24:48,2,1.03,1.0,N,229,140,1,...,Manhattan,Lenox Hill East,-73.954739,40.765484,short,3.666667,5.0,0.0,0.0,1


## 2. Создание признаков

KNN чувствителен к масштабу признаков, поэтому числовые признаки дальше стандартизируются. Категориальные признаки кодируются через one-hot encoding.

In [10]:
print("Создаем признаки для KNN...")

model_df = df.copy()

model_df["store_and_fwd_flag"] = model_df["store_and_fwd_flag"].fillna("unknown").astype(str)
model_df["distance_group"] = model_df["distance_group"].astype(str).replace("nan", "unknown")
model_df["PU_Borough"] = model_df["PU_Borough"].fillna("unknown").astype(str)
model_df["DO_Borough"] = model_df["DO_Borough"].fillna("unknown").astype(str)

model_df["pickup_hour"] = model_df["tpep_pickup_datetime"].dt.hour
model_df["pickup_dayofweek"] = model_df["tpep_pickup_datetime"].dt.dayofweek
model_df["pickup_month"] = model_df["tpep_pickup_datetime"].dt.month
model_df["pickup_day"] = model_df["tpep_pickup_datetime"].dt.day
model_df["pickup_weekofyear"] = model_df["tpep_pickup_datetime"].dt.isocalendar().week.astype(int)
model_df["is_weekend"] = model_df["pickup_dayofweek"].isin([5, 6]).astype(int)
model_df["is_rush_hour"] = (
    (model_df["pickup_dayofweek"] < 5)
    & (model_df["pickup_hour"] >= 16)
    & (model_df["pickup_hour"] < 20)
).astype(int)
model_df["is_night_tariff"] = ((model_df["pickup_hour"] >= 20) | (model_df["pickup_hour"] < 6)).astype(int)

model_df["hour_sin"] = np.sin(2 * np.pi * model_df["pickup_hour"] / 24)
model_df["hour_cos"] = np.cos(2 * np.pi * model_df["pickup_hour"] / 24)
model_df["dow_sin"] = np.sin(2 * np.pi * model_df["pickup_dayofweek"] / 7)
model_df["dow_cos"] = np.cos(2 * np.pi * model_df["pickup_dayofweek"] / 7)
model_df["month_sin"] = np.sin(2 * np.pi * model_df["pickup_month"] / 12)
model_df["month_cos"] = np.cos(2 * np.pi * model_df["pickup_month"] / 12)

model_df["lat_diff"] = (model_df["DO_lat"] - model_df["PU_lat"]).abs()
model_df["lon_diff"] = (model_df["DO_lon"] - model_df["PU_lon"]).abs()
model_df["gps_distance"] = model_df["lat_diff"] + model_df["lon_diff"]
model_df["distance_ratio"] = model_df["trip_distance"] / (model_df["gps_distance"] + 0.001)
model_df["distance_ratio"] = model_df["distance_ratio"].clip(0, 15)

model_df["duration_min"] = model_df["duration_min"].clip(lower=0.1)
model_df["speed_mph"] = model_df["trip_distance"] / (model_df["duration_min"] / 60 + 0.01)
model_df["speed_mph"] = model_df["speed_mph"].clip(0, 80)
model_df["log_trip_distance"] = np.log1p(model_df["trip_distance"])
model_df["log_duration_min"] = np.log1p(model_df["duration_min"])

model_df["has_precipitation"] = (model_df["precipitation"] > 0).astype(int)
model_df["has_snowfall"] = (model_df["snowfall"] > 0).astype(int)
model_df["bad_weather"] = ((model_df["precipitation"] > 0) | (model_df["snowfall"] > 0)).astype(int)
model_df["precipitation_x_rush_hour"] = model_df["precipitation"] * model_df["is_rush_hour"]
model_df["snowfall_x_rush_hour"] = model_df["snowfall"] * model_df["is_rush_hour"]

airport_zone_ids = {1, 132, 138}
model_df["pickup_airport"] = model_df["PULocationID"].isin(airport_zone_ids).astype(int)
model_df["dropoff_airport"] = model_df["DOLocationID"].isin(airport_zone_ids).astype(int)
model_df["airport_trip"] = ((model_df["pickup_airport"] == 1) | (model_df["dropoff_airport"] == 1)).astype(int)
model_df["same_zone"] = (model_df["PULocationID"] == model_df["DOLocationID"]).astype(int)
model_df["same_borough"] = (model_df["PU_Borough"] == model_df["DO_Borough"]).astype(int)
model_df["interborough_trip"] = (model_df["PU_Borough"] != model_df["DO_Borough"]).astype(int)
model_df["route_id"] = model_df["PULocationID"].astype(str) + "_" + model_df["DOLocationID"].astype(str)

print("Признаки созданы")

Создаем признаки для KNN...
Признаки созданы


## 3. Разбиение на обучение/тест и target encoding

Разбиение делается по времени: train до `2024-10-01`, test после этой даты. Если в датасете нет такого периода, используется fallback 80/20. Target encoding считается только на train, чтобы не было утечки таргета из test.

In [11]:
categorical_cols_for_knn = [
    "VendorID",
    "RatecodeID",
    "payment_type",
    "store_and_fwd_flag",
    "PU_Borough",
    "DO_Borough",
    "distance_group",
]

target_encoding_cols = [
    "PULocationID",
    "DOLocationID",
    "route_id",
    "RatecodeID",
]

target_encoded_feature_names = [
    "PU_zone_average_price",
    "DO_zone_average_price",
    "route_average_price",
    "ratecode_average_price",
]

weather_cols = [
    "temperature",
    "precipitation",
    "snowfall",
    "weather_code",
    "has_precipitation",
    "has_snowfall",
    "bad_weather",
    "precipitation_x_rush_hour",
    "snowfall_x_rush_hour",
]

time_cols = [
    "pickup_hour",
    "pickup_dayofweek",
    "pickup_month",
    "pickup_day",
    "pickup_weekofyear",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos",
]

trip_cols = [
    "trip_distance",
    "passenger_count",
    "is_weekend",
    "is_rush_hour",
    "is_night_tariff",
    "gps_distance",
    "distance_ratio",
    "duration_min",
    "speed_mph",
    "log_trip_distance",
    "log_duration_min",
    "pickup_airport",
    "dropoff_airport",
    "airport_trip",
    "same_zone",
    "same_borough",
    "interborough_trip",
]

numerical_cols_for_knn = [
    *trip_cols,
    *time_cols,
    *weather_cols,
    *target_encoded_feature_names,
]

base_feature_cols = [
    "PULocationID",
    "DOLocationID",
    "route_id",
    "tpep_pickup_datetime",
    *categorical_cols_for_knn,
    *[col for col in numerical_cols_for_knn if col not in target_encoded_feature_names],
]

model_df = model_df.dropna(subset=[TARGET_COL, *base_feature_cols]).copy()
model_df = model_df.sort_values("tpep_pickup_datetime").reset_index(drop=True)

test_start = pd.Timestamp("2024-10-01")
train_mask = model_df["tpep_pickup_datetime"] < test_start
test_mask = model_df["tpep_pickup_datetime"] >= test_start

if test_mask.sum() == 0 or train_mask.sum() == 0:
    split_idx = int(len(model_df) * 0.8)
    train_mask = model_df.index < split_idx
    test_mask = model_df.index >= split_idx
    print("Резервное разбиение: первые 80% данных - обучение, последние 20% - тест")
else:
    print("Временное разбиение: обучение до", test_start.date(), "тест с", test_start.date())

X_train_base = model_df.loc[train_mask, base_feature_cols].drop(columns=["tpep_pickup_datetime"]).copy()
X_test_base = model_df.loc[test_mask, base_feature_cols].drop(columns=["tpep_pickup_datetime"]).copy()
y_train = model_df.loc[train_mask, TARGET_COL].copy()
y_test = model_df.loc[test_mask, TARGET_COL].copy()

def add_target_encoding(train_x, test_x, train_y, source_cols, encoded_cols, smooth=50):
    train_x = train_x.copy()
    test_x = test_x.copy()
    global_mean = float(train_y.mean())

    train_with_target = train_x.copy()
    train_with_target[TARGET_COL] = train_y.values

    for source_col, encoded_col in zip(source_cols, encoded_cols):
        stats = train_with_target.groupby(source_col, observed=False)[TARGET_COL].agg(["mean", "count"])
        smoothed = (stats["mean"] * stats["count"] + global_mean * smooth) / (stats["count"] + smooth)
        train_x[encoded_col] = train_x[source_col].map(smoothed).fillna(global_mean).astype(float)
        test_x[encoded_col] = test_x[source_col].map(smoothed).fillna(global_mean).astype(float)

    return train_x, test_x

X_train_full, X_test_full = add_target_encoding(
    X_train_base,
    X_test_base,
    y_train,
    target_encoding_cols,
    target_encoded_feature_names,
)

X_train_knn = X_train_full[numerical_cols_for_knn + categorical_cols_for_knn].copy()
X_test_knn = X_test_full[numerical_cols_for_knn + categorical_cols_for_knn].copy()
y_train_knn = y_train.copy()
y_test_knn = y_test.copy()

print("Размер полной обучающей выборки KNN:", X_train_knn.shape)
print("Размер полной тестовой выборки KNN:", X_test_knn.shape)

Резервное разбиение: первые 80% данных - обучение, последние 20% - тест
Размер полной обучающей выборки KNN: (80000, 48)
Размер полной тестовой выборки KNN: (20000, 48)


## 4. Подвыборка для KNN

KNN медленно предсказывает на больших данных, потому что для каждого тестового объекта ищет ближайших соседей среди train. Для улучшения качества используется весь доступный train до 80 000 строк, а test ограничивается 5 000 строками. Также ниже выбирается более компактный набор признаков: для KNN это важно, потому что лишние слабые признаки ухудшают расстояния между объектами.

In [12]:
optimized_numerical_cols_for_knn = [
    "trip_distance",
    "duration_min",
    "log_trip_distance",
    "log_duration_min",
    "speed_mph",
    "gps_distance",
    "distance_ratio",
    "pickup_airport",
    "dropoff_airport",
    "airport_trip",
    "same_zone",
    "same_borough",
    "interborough_trip",
    "pickup_hour",
    "is_weekend",
    "is_rush_hour",
    "is_night_tariff",
    "PU_zone_average_price",
    "DO_zone_average_price",
    "route_average_price",
    "ratecode_average_price",
]

optimized_feature_cols_for_knn = optimized_numerical_cols_for_knn + categorical_cols_for_knn

train_sample_size = min(80_000, len(X_train_knn))
test_sample_size = min(5_000, len(X_test_knn))

train_idx = X_train_knn.sample(train_sample_size, random_state=42).index
test_idx = X_test_knn.sample(test_sample_size, random_state=42).index

X_train_knn_sample = X_train_knn.loc[train_idx, optimized_feature_cols_for_knn].copy()
y_train_knn_sample = y_train_knn.loc[train_idx].copy()
X_test_knn_sample = X_test_knn.loc[test_idx, optimized_feature_cols_for_knn].copy()
y_test_knn_sample = y_test_knn.loc[test_idx].copy()

print("Количество отобранных признаков KNN:", len(optimized_feature_cols_for_knn))
print("Размер обучающей подвыборки KNN:", X_train_knn_sample.shape)
print("Размер тестовой подвыборки KNN:", X_test_knn_sample.shape)

Количество отобранных признаков KNN: 28
Размер обучающей подвыборки KNN: (80000, 28)
Размер тестовой подвыборки KNN: (5000, 28)


## 5. Обучение KNN

Числовые признаки масштабируются через `StandardScaler`, категориальные признаки кодируются через `OneHotEncoder`, после чего обучается `KNeighborsRegressor`. Для текущих данных лучше сработали `n_neighbors=9` и Manhattan distance: эта метрика оказалась устойчивее евклидовой на смешанных признаках после кодирования.

In [13]:
try:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), optimized_numerical_cols_for_knn),
        ("cat", one_hot_encoder, categorical_cols_for_knn),
    ]
)

knn_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", KNeighborsRegressor(
        n_neighbors=9,
        weights="distance",
        metric="manhattan",
        n_jobs=1,
    )),
])

print("Обучаем pipeline KNN...")
start_time = time.time()
knn_pipeline.fit(X_train_knn_sample, y_train_knn_sample)
print(f"Модель обучена за {time.time() - start_time:.1f} сек.")

print("Строим прогноз на тестовой подвыборке...")
pred_start = time.time()
y_pred_knn = knn_pipeline.predict(X_test_knn_sample)
print(f"Прогноз построен за {time.time() - pred_start:.1f} сек.")

Обучаем pipeline KNN...
Модель обучена за 0.1 сек.
Строим прогноз на тестовой подвыборке...
Прогноз построен за 12.1 сек.


## 6. Метрики качества

Метрики считаются на тестовой подвыборке. MAE и RMSE измеряются в долларах, R2 показывает долю объясненной вариации целевой переменной.

In [14]:
mae_knn = mean_absolute_error(y_test_knn_sample, y_pred_knn)
rmse_knn = np.sqrt(mean_squared_error(y_test_knn_sample, y_pred_knn))
r2_knn = r2_score(y_test_knn_sample, y_pred_knn)

print("=== Результаты K-ближайших соседей ===")
print(f"MAE:  {mae_knn:.4f} долл. США")
print(f"RMSE: {rmse_knn:.4f} долл. США")
print(f"R2:   {r2_knn:.4f}")

=== Результаты K-ближайших соседей ===
MAE:  1.6723 долл. США
RMSE: 3.1565 долл. США
R2:   0.9719
